# HR Analytics: Job Change of Data Scientists

**Course:** Introduction to Data Science  
**Business problem:** Predict whether a data-science training enrollee is looking for a job change so HR/Talent Acquisition can prioritise outreach and training-to-hire spend.

**Dataset:** Kaggle — HR Analytics: Job Change of Data Scientists by arashnic.

---

## A1. Business Understanding

### Stakeholder
The stakeholder is the **HR / Talent Acquisition team**.

### Decision supported
The model supports the decision: **Which trained candidates should HR prioritise for recruitment outreach?**

### What success looks like
A useful system should identify genuine job seekers without creating excessive unnecessary outreach. Because the positive class is the minority, accuracy alone is not sufficient; F1, recall, precision and ROC-AUC are more informative.

### Why this is classification
The target has two labelled outcomes:
- `0` = not looking for a job change
- `1` = looking for a job change

This is therefore a **binary classification** problem, not regression or clustering.

In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# Works whether Jupyter was launched from repo root or notebooks/
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "aug_train.csv"
OUTPUT_DIR = ROOT / "outputs"
ARTIFACT_DIR = ROOT / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Repository root:", ROOT)
print("Expected data file:", DATA_PATH)

### What I did and why
I imported the libraries needed for data handling, visualisation, preprocessing, modelling and evaluation. `RANDOM_STATE = 42` makes random operations reproducible. All output paths are created once so the notebook can be run from top to bottom without changing filenames manually.

## A2. Data Understanding and Exploratory Data Analysis

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Download aug_train.csv from Kaggle and place it in data/."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

### What I did and why
I first inspected shape, data types, summary statistics and duplicate rows instead of jumping directly into modelling. This identifies which variables are genuinely numeric and which only look numeric conceptually. In particular, `experience` and `last_new_job` are stored as text because they contain values such as `<1`, `>20`, `never` and `>4`.

### Missing-value analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percentage", ascending=False)

display(missing_summary)

missing_plot = missing_summary[missing_summary["Missing_Count"] > 0].sort_values(
    "Missing_Percentage"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=missing_plot.reset_index(),
    x="Missing_Percentage",
    y="index"
)
plt.title("Percentage of Missing Values by Feature")
plt.xlabel("Missing values (%)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missing-Value Pattern")
plt.xlabel("Features")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

### Missing-value takeaway
Several categorical variables have meaningful levels of missingness. I do **not** drop every incomplete row because that would discard a large amount of information and could create selection bias. Categorical missing values will be represented as `Unknown`, while numerical missing values will be imputed with the median inside the modelling pipeline.

### Target class balance

In [ ]:
target_summary = pd.DataFrame({
    "Count": df["target"].value_counts().sort_index(),
    "Percentage": (
        df["target"].value_counts(normalize=True).sort_index() * 100
    ).round(2)
})

display(target_summary)

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="target")
plt.title("Target Class Distribution")
plt.xlabel("Job-change status")
plt.ylabel("Number of candidates")
plt.xticks([0, 1], ["Not looking (0)", "Looking (1)"])
plt.tight_layout()
plt.show()

### Why accuracy alone is misleading
The positive class is substantially smaller than the negative class. A model could predict `0` for almost everyone and still appear accurate while missing the people HR wants to identify. For this reason, model comparison focuses on **F1 and ROC-AUC**, with recall and precision also reported for the final model.

### Numeric distributions

In [ ]:
numeric_eda_cols = ["city_development_index", "training_hours"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for col, ax in zip(numeric_eda_cols, axes):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for col, ax in zip(numeric_eda_cols, axes):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_title(f"{col} by Target")
plt.tight_layout()
plt.show()

### Numeric-variable takeaway
The numerical predictors operate on different scales, which matters for models such as Logistic Regression and KNN. For that reason, numerical features will later be standardized. The plots are descriptive and do not establish causal relationships.

### Categorical breakdowns of target rate

In [ ]:
def plot_target_rate(data, column, order=None):
    temp = data[[column, "target"]].copy()
    temp[column] = temp[column].fillna("Missing")

    summary = (
        temp.groupby(column, observed=False)["target"]
        .agg(["mean", "count"])
        .reset_index()
    )
    summary["target_rate_pct"] = summary["mean"] * 100

    if order is not None:
        present = [x for x in order if x in summary[column].astype(str).values]
        remainder = [x for x in summary[column].astype(str).values if x not in present]
        summary[column] = pd.Categorical(
            summary[column],
            categories=present + remainder,
            ordered=True
        )
        summary = summary.sort_values(column)

    plt.figure(figsize=(10, 5))
    sns.barplot(data=summary, x=column, y="target_rate_pct")
    plt.title(f"Job-Change Rate by {column}")
    plt.ylabel("Looking for job change (%)")
    plt.xlabel(column)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return summary[[column, "count", "target_rate_pct"]]

In [ ]:
education_order = [
    "Primary School", "High School", "Graduate", "Masters", "Phd", "Missing"
]
display(plot_target_rate(df, "education_level", education_order))

**Education takeaway:** target rates differ across education groups, so education may contain predictive signal. This is an association in the sample, not evidence that education level itself causes job searching.

In [ ]:
experience_order = ["<1"] + [str(i) for i in range(1, 21)] + [">20", "Missing"]
display(plot_target_rate(df, "experience", experience_order))

**Experience takeaway:** job-change rates vary across career stages. Experience is therefore a reasonable candidate predictor, although individual categories with small counts should not be over-interpreted.

In [ ]:
company_size_order = [
    "<10", "10/49", "50-99", "100-500",
    "500-999", "1000-4999", "5000-9999", "10000+", "Missing"
]
display(plot_target_rate(df, "company_size", company_size_order))

**Company-size takeaway:** target rates are not identical across employer-size groups. Because company size also has substantial missingness, retaining an explicit missing category is preferable to deleting those candidates.

In [ ]:
display(plot_target_rate(df, "company_type"))

**Company-type takeaway:** job-search behaviour varies across employer types. This may provide useful predictive information, while the amount of missing company-type data remains an important limitation.

In [ ]:
display(plot_target_rate(df, "relevent_experience"))

**Relevant-experience takeaway:** candidates with and without relevant experience show different target rates. This variable directly describes prior experience in the field and may help distinguish different candidate profiles.

### Correlation view

In [ ]:
corr_matrix = df[
    ["city_development_index", "training_hours", "target"]
].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix for Numerical Variables")
plt.tight_layout()
plt.show()

### Correlation takeaway
I only calculate ordinary correlation where numerical distance is meaningful. I do not assign arbitrary integer codes to nominal categories just to make a larger heatmap, because those numbers would create artificial relationships. Correlation also indicates association, not causation.

## A3. Data Preparation

In [ ]:
X = df.drop(columns=["target"]).copy()
y = df["target"].astype(int).copy()

# Unique identifier, not a behavioural predictor
X = X.drop(columns=["enrollee_id"])

experience_mapping = {
    "<1": 0.5,
    **{str(i): i for i in range(1, 21)},
    ">20": 21
}

last_job_mapping = {
    "never": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    ">4": 5
}

X["experience_num"] = X["experience"].map(experience_mapping)
X["last_new_job_num"] = X["last_new_job"].map(last_job_mapping)

display(
    X[["experience", "experience_num"]]
    .drop_duplicates()
    .sort_values("experience_num")
)

display(
    X[["last_new_job", "last_new_job_num"]]
    .drop_duplicates()
    .sort_values("last_new_job_num")
)

X = X.drop(columns=["experience", "last_new_job"])

### Mapping assumptions
`<1 → 0.5` represents experience between zero and one year. `>20 → 21` and `>4 → 5` preserve ordering but do **not** claim that every person in those categories has exactly 21 or 5 years. These are transparent practical encodings for ordered text categories.

`enrollee_id` is removed because it is an identifier rather than a meaningful candidate characteristic.

### Train/test split before fitting preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).sort_index())

print("\nTest target proportions:")
display(y_test.value_counts(normalize=True).sort_index())

### Why stratify?
The target is imbalanced, so stratification keeps approximately the same positive/negative proportion in both sets. No imputer, scaler or encoder is fitted before this split; those steps are learned inside the training pipeline, which prevents test leakage.

In [ ]:
nominal_features = [
    "city",
    "gender",
    "relevent_experience",
    "enrolled_university",
    "major_discipline",
    "company_type"
]

ordinal_features = [
    "education_level",
    "company_size"
]

numeric_features = [
    "city_development_index",
    "training_hours",
    "experience_num",
    "last_new_job_num"
]

education_categories = [
    "Unknown",
    "Primary School",
    "High School",
    "Graduate",
    "Masters",
    "Phd"
]

company_size_categories = [
    "Unknown",
    "<10",
    "10/49",
    "50-99",
    "100-500",
    "500-999",
    "1000-4999",
    "5000-9999",
    "10000+"
]

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("ordinal", OrdinalEncoder(
        categories=[education_categories, company_size_categories],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("nominal", nominal_pipeline, nominal_features),
    ("ordinal", ordinal_pipeline, ordinal_features),
    ("numeric", numeric_pipeline, numeric_features)
], remainder="drop")

### Encoding and scaling choices
- **One-Hot Encoding** is used for nominal variables because their categories have no meaningful numerical ranking.
- **Ordinal Encoding** is used for education and company size because they have a natural order.
- `experience` and `last_new_job` use explicit numeric mappings because those mappings are easy to explain.
- **StandardScaler** is used for numerical variables because Logistic Regression and KNN are scale-sensitive.
- All preprocessing sits inside a `ColumnTransformer` inside each model `Pipeline`, so fitting happens only on training folds.

## A4. Modelling and 5-Fold Stratified Cross-Validation

In [ ]:
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=15
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "f1": "f1",
    "roc_auc": "roc_auc"
}

model_pipelines = {}
cv_rows = []

for model_name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    model_pipelines[model_name] = pipe

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_rows.append({
        "Model": model_name,
        "Mean_F1": scores["test_f1"].mean(),
        "Std_F1": scores["test_f1"].std(),
        "Mean_ROC_AUC": scores["test_roc_auc"].mean(),
        "Std_ROC_AUC": scores["test_roc_auc"].std()
    })

cv_results_df = (
    pd.DataFrame(cv_rows)
    .sort_values("Mean_ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

display(cv_results_df)
cv_results_df.to_csv(OUTPUT_DIR / "cv_model_comparison.csv", index=False)

In [ ]:
comparison_plot = cv_results_df.melt(
    id_vars="Model",
    value_vars=["Mean_F1", "Mean_ROC_AUC"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 6))
sns.barplot(data=comparison_plot, x="Model", y="Score", hue="Metric")
plt.ylim(0, 1)
plt.title("5-Fold Cross-Validation Model Comparison")
plt.ylabel("Mean CV score")
plt.xlabel("Model")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

### Why these models?
The DummyClassifier is a minimum baseline. Logistic Regression is a simple interpretable linear model. Decision Tree can capture nonlinear splits and interactions. Random Forest combines many trees to reduce the instability of a single tree. KNN gives a distance-based comparison and demonstrates why scaling matters.

Five-fold **StratifiedKFold** keeps class proportions similar across folds. F1 balances precision and recall for the positive class, while ROC-AUC evaluates ranking quality across possible thresholds.

## A5. Model Selection and Hyperparameter Tuning

In [ ]:
best_model_name = cv_results_df.iloc[0]["Model"]
print("Best untuned model by mean CV ROC-AUC:", best_model_name)

if best_model_name == "Dummy":
    raise RuntimeError(
        "Dummy baseline ranked first. Review preprocessing and model configuration."
    )

parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"]
    },

    "Decision Tree": {
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_split": [2, 10, 30],
        "model__min_samples_leaf": [1, 5, 15]
    },

    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [6, 10, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__max_features": ["sqrt", 0.5]
    },

    "KNN": {
        "model__n_neighbors": [7, 15, 25, 35],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    }
}

grid_search = GridSearchCV(
    estimator=model_pipelines[best_model_name],
    param_grid=parameter_grids[best_model_name],
    scoring={"f1": "f1", "roc_auc": "roc_auc"},
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid_search.fit(X_train, y_train)

print("Tuned model:", best_model_name)
print("Best parameters:", grid_search.best_params_)

### Selection and tuning assumption
I use **mean cross-validated ROC-AUC as the primary model-selection metric**, with F1 as a supporting metric. This fits a workflow where HR ranks candidates by predicted likelihood and may choose different outreach thresholds.

The tuning grid is intentionally small and defensible for a course project. It demonstrates correct hyperparameter tuning without pretending that an exhaustive global search was performed.

In [ ]:
before_row = cv_results_df.loc[
    cv_results_df["Model"] == best_model_name
].iloc[0]

best_idx = grid_search.best_index_

tuning_comparison = pd.DataFrame({
    "Stage": ["Before tuning", "After tuning"],
    "Mean CV F1": [
        before_row["Mean_F1"],
        grid_search.cv_results_["mean_test_f1"][best_idx]
    ],
    "Mean CV ROC-AUC": [
        before_row["Mean_ROC_AUC"],
        grid_search.cv_results_["mean_test_roc_auc"][best_idx]
    ]
})

display(tuning_comparison)
tuning_comparison.to_csv(OUTPUT_DIR / "tuning_comparison.csv", index=False)

plot_df = tuning_comparison.melt(
    id_vars="Stage",
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(8, 5))
sns.barplot(data=plot_df, x="Stage", y="Score", hue="Metric")
plt.ylim(0, 1)
plt.title(f"{best_model_name}: Before vs After Tuning")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tuning_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## A6. Evaluation on the Held-Out Test Set

In [ ]:
final_model = grid_search.best_estimator_

y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob)
    ]
})

display(test_metrics)
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)

print(classification_report(y_test, y_pred, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", ax=ax
)
plt.title("Confusion Matrix – Held-Out Test Set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC Curve – Held-Out Test Set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve.png", dpi=200, bbox_inches="tight")
plt.show()

### Which metric matters most?
For HR outreach, **recall for class 1 is especially important** because a false negative is a genuine job seeker whom HR misses. However, maximizing recall alone could create too many false positives and waste recruiter time. F1 is useful when balancing these costs, while ROC-AUC is useful when HR ranks candidates and chooses its own threshold.

### Precision–recall trade-off by probability threshold

In [ ]:
threshold_rows = []

for threshold in np.arange(0.20, 0.81, 0.05):
    threshold_pred = (y_prob >= threshold).astype(int)

    threshold_rows.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(y_test, threshold_pred, zero_division=0),
        "Recall": recall_score(y_test, threshold_pred, zero_division=0),
        "F1": f1_score(y_test, threshold_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_df.to_csv(OUTPUT_DIR / "threshold_tradeoff.csv", index=False)

ax = threshold_df.plot(
    x="Threshold",
    y=["Precision", "Recall", "F1"],
    figsize=(9, 5),
    marker="o"
)
ax.set_title("Precision-Recall Trade-off by Classification Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "threshold_tradeoff.png", dpi=200, bbox_inches="tight")
plt.show()

The default threshold of 0.50 is not a magical business rule. Lowering the threshold usually increases recall but reduces precision; raising it usually does the opposite. HR should choose the threshold using the actual cost of missed job seekers versus unnecessary outreach.

### Feature importance

In [ ]:
# Model-agnostic importance at original-feature level
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(importance_df.head(10))
importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

top_features = importance_df.head(10).sort_values("Importance_Mean")

plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance_Mean"])
plt.xlabel("Decrease in ROC-AUC when feature is shuffled")
plt.ylabel("Feature")
plt.title("Permutation Feature Importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

### Interpreting the top features
Permutation importance measures how much ROC-AUC falls when one feature is shuffled. A larger reduction means the fitted model relied more on that feature. After running the notebook, use the first five rows of the table above as the slide's top-five features.

These are **predictive associations**, not causal effects. We should not claim that changing one of these attributes would cause a candidate to start or stop job searching.

### Save the fitted pipeline

In [ ]:
with open(ARTIFACT_DIR / "final_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Saved fitted pipeline to:", ARTIFACT_DIR / "final_model.pkl")

## A7. Conclusions, Recommendations and Limitations

### Business recommendation
Use the model as a **prioritisation tool**, not an automatic recruitment or hiring decision.

A practical workflow is:
1. Score eligible trainees.
2. Rank them by predicted probability of looking for a job change.
3. Select an outreach threshold based on recruiter capacity.
4. Track recruiter response, interview and hiring outcomes.
5. Re-evaluate the model as candidate behaviour and labour-market conditions change.

If HR wants to avoid missing genuine job seekers, it can lower the threshold to increase recall. If recruiter capacity is limited, it can raise the threshold to improve precision.

### Next steps
Useful additional variables could include:
- current and expected salary,
- desired role,
- remote/hybrid preference,
- geographic mobility,
- reason for completing training,
- job applications submitted,
- recruiter-response history,
- time since course completion,
- actual interview and hiring outcomes.

### Honest limitations
1. **Class imbalance:** the positive class is much smaller than the negative class.
2. **Missing data:** several important profile variables are incomplete.
3. **Limited explanatory variables:** the data does not directly capture why a candidate wants to change jobs.
4. **Geographic representation:** performance may not generalise equally to other labour markets.
5. **Historical data:** candidate behaviour may drift over time.
6. **Correlation ≠ causation:** predictive importance is not causal evidence.
7. **Fairness:** gender exists in the source data; a production HR system requires fairness and legal/policy review.
8. **Target scope:** looking for a job change is not the same as being qualified for a particular vacancy.

## Final presentation checklist

Before making the final slides:
- copy the actual CV table from `cv_results_df`,
- copy the selected model and best parameters,
- copy the held-out test metrics,
- add the confusion matrix and ROC curve,
- copy the actual top five features,
- do **not** invent scores before running the notebook,
- rename `GroupNo` to your actual group number.